# Exploração dos dados processados do IBGE (Porto Alegre)

Verificação simples dos arquivos gerados por `scripts/limpeza_ibge.py` em `dados/processados/IBGE/`: ligação dos 94 bairros do INCT ao IBGE, população por bairro e renda por área de ponderação.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

PASTA = Path('..', '..') / 'dados' / 'processados' / 'IBGE'

bairros = pd.read_csv(PASTA / 'bairros_poa.csv', dtype={'cd_bairro_ibge': str, 'areas': str, 'codigo_area': str})
setores = pd.read_csv(PASTA / 'setores_bairros_poa.csv', dtype={'setor': str, 'cd_bairro': str})
areas = pd.read_csv(PASTA / 'areas_ponderacao_poa.csv')

print('bairros:', bairros.shape, '| setores:', setores.shape, '| áreas:', areas.shape)
bairros.head()

bairros: (94, 14) | setores: (2744, 4) | áreas: (46, 13)


,bairro_inct,populacao_inct,status,cd_bairro_ibge,nome_bairro_ibge,setores,populacao_ibge,areas,n_areas,codigo_area,nome_area,renda_media_total,renda_mediana_total,metodo_renda
0,Aberta dos Morros,9854,ok,4314902080,Aberta dos Morros,12,9854,4314902027,1,4314902027,"Serraria, Aberta dos Morros e Hípica",2283.75,1600.0,área de ponderação
1,Agronomia,2677,ok,4314902035,Agronomia,8,2677,4314902039,1,4314902039,Agronomia e Lomba Norte,1699.82,975.0,área de ponderação
2,Anchieta,791,ok,4314902055,Anchieta,6,791,4314902007,1,4314902007,"São Geraldo, Navegantes, Humaitá e Anchieta",2770.43,1930.0,área de ponderação
3,Arquipélago,6411,ok,4314902033,Arquipélago,28,6411,4314902006,1,4314902006,Arquipélago e Farrapos,1318.86,925.0,área de ponderação
4,Auxiliadora,8909,ok,4314902045,Auxiliadora,21,8909,4314902032,1,4314902032,Moinhos de Vento e Auxiliadora,10110.21,5500.0,área de ponderação


## Ligação dos bairros do INCT ao IBGE

In [2]:
print(bairros['status'].value_counts().to_string())
print()
print('População IBGE = INCT em todos os bairros:', bool((bairros['populacao_ibge'] == bairros['populacao_inct']).all()))
print('Soma da população (IBGE):', int(setores['populacao'].sum()), '| (INCT):', int(bairros['populacao_inct'].sum()))
print('Setores por bairro: min', setores.groupby('cd_bairro').size().min(), '| max', setores.groupby('cd_bairro').size().max())
bairros[bairros['status'] != 'ok'][['bairro_inct', 'status', 'areas']]

status
ok                    89
ok_media_ponderada     5

População IBGE = INCT em todos os bairros: True
Soma da população (IBGE): 1332845 | (INCT): 1332845
Setores por bairro: min 2 | max 117


,bairro_inct,status,areas
50,Lomba do Pinheiro,ok_media_ponderada,4314902039; 4314902047
60,Partenon,ok_media_ponderada,4314902037; 4314902045
64,Petrópolis,ok_media_ponderada,4314902034; 4314902042
68,Restinga,ok_media_ponderada,4314902029; 4314902030
82,Sarandi,ok_media_ponderada,4314902013; 4314902036


## Renda por bairro

Os 89 bairros com uma área recebem a renda dessa área. Os 5 bairros em duas áreas recebem a média ponderada pela população dos setores em cada área (ver a seção 5 do relatório de limpeza).

In [3]:
com_renda = bairros.dropna(subset=['renda_mediana_total']).sort_values('renda_mediana_total')
fig = px.bar(com_renda, x='renda_mediana_total', y='bairro_inct', orientation='h',
             labels={'renda_mediana_total': 'Renda domiciliar per capita mediana (R$)', 'bairro_inct': ''},
             title='Renda mediana por bairro (área de ponderação)', height=2200)
fig.show()
print('bairros com renda:', len(com_renda), '| sem renda:', int(bairros['renda_mediana_total'].isna().sum()))

bairros com renda: 94 | sem renda: 0


**Comentário.** A renda mediana per capita vai de R$ 882,40 (Pitinga) a R$ 7.250,00 (Bela Vista), e o valor típico entre os bairros é R$ 1.940. Bairros da mesma área de ponderação aparecem com o mesmo valor, porque a renda vem da área, e não do bairro.

## População: INCT × IBGE

Os pontos devem estar sobre a diagonal.

In [4]:
fig = px.scatter(bairros, x='populacao_inct', y='populacao_ibge', hover_name='bairro_inct',
                 labels={'populacao_inct': 'População (INCT)', 'populacao_ibge': 'População (IBGE)'},
                 title='População por bairro: INCT × IBGE')
fig.show()

**Comentário.** Todos os pontos ficam sobre a diagonal: a população de cada bairro no INCT é igual à soma dos setores do IBGE. Isso confirma que a ligação entre as duas bases está correta e que a população do INCT vem do Censo 2022.

## Áreas de ponderação

In [5]:
print('valores de renda ausentes nas áreas:', int(areas[['renda_media_total', 'renda_mediana_total']].isna().sum().sum()))
print('população nas áreas:', int(areas['pessoas_total'].sum()))
areas[['codigo_area', 'nome_area', 'pessoas_total', 'renda_media_total', 'renda_mediana_total']].head(10)

valores de renda ausentes nas áreas: 0
população nas áreas: 1321927


,codigo_area,nome_area,pessoas_total,renda_media_total,renda_mediana_total
0,4314902001,"Floresta, Independência, Bom Fim e Farroupilha",26334,4903.91,3750.00
1,4314902002,Cidade Baixa e Azenha,24802,4439.59,3333.33
2,4314902003,Santana,17594,4518.78,3300.00
3,4314902004,Rio Branco e Santa Cecília,20178,7557.36,5333.33
4,4314902005,Mont Serrat e Bela Vista,22040,10133.89,7250.00
5,4314902006,Arquipélago e Farrapos,24002,1318.86,925.00
6,4314902007,"São Geraldo, Navegantes, Humaitá e Anchieta",23541,2770.43,1930.00
7,4314902008,"São João, Santa Maria Goretti, São Pedro e Jar...",18941,4125.18,3000.00
8,4314902009,"Cristo Redentor, Jardim Lindóia e São Sebastião",29960,4168.11,3166.67
9,4314902010,Vila Ipiranga,17846,4294.00,2625.00


## Exemplo: renda comprometida em 5,0 m, por bairro

Renda comprometida = população afetada pela enchente no nível de 5,0 m × renda média mensal per capita do bairro (IBGE). É uma estimativa: supõe que a renda média do bairro vale para as pessoas afetadas. Dados do INCT limpos, nível 5,0 m.

In [6]:
import json

with open(Path('..', '..') / 'dados' / 'processados' / 'INCT' / 'inct_limpo.json', encoding='utf-8') as f:
    inct = pd.DataFrame(json.load(f))

nivel = inct[inct['nivel_m'] == 5.0][['bairro', 'population', 'affected_population']]
exemplo = nivel.merge(bairros[['bairro_inct', 'renda_media_total']],
                      left_on='bairro', right_on='bairro_inct', how='left')
exemplo['renda_comprometida'] = exemplo['affected_population'] * exemplo['renda_media_total']
exemplo['pct_populacao_afetada'] = 100 * exemplo['affected_population'] / exemplo['population']
exemplo = exemplo.dropna(subset=['renda_comprometida']).sort_values('renda_comprometida', ascending=False)

print('bairros:', len(exemplo), '| renda comprometida total (R$ milhões/mês):',
      round(exemplo['renda_comprometida'].sum() / 1e6, 1))
exemplo.head()

bairros: 94 | renda comprometida total (R$ milhões/mês): 384.7


,bairro,population,affected_population,bairro_inct,renda_media_total,renda_comprometida,pct_populacao_afetada
53,Menino Deus,27961,15099,Menino Deus,5946.530000,8.978666e+07,54.000215
81,Sarandi,51539,21131,Sarandi,2144.126892,4.530755e+07,41.000019
34,Humaitá,12744,12744,Humaitá,2770.430000,3.530636e+07,100.000000
29,Floresta,8798,5543,Floresta,4903.910000,2.718237e+07,63.002955
20,Cidade Baixa,13014,5857,Cidade Baixa,4439.590000,2.600268e+07,45.005379


In [7]:
fig = px.bar(exemplo.sort_values('renda_comprometida'), x='renda_comprometida', y='bairro',
             orientation='h', hover_data={'pct_populacao_afetada': ':.1f', 'renda_media_total': ':.2f',
                                          'affected_population': True},
             labels={'renda_comprometida': 'Renda mensal comprometida (R$)', 'bairro': ''},
             title='Renda mensal comprometida em 5,0 m, por bairro', height=2200)
fig.show()

**Comentário.** Em 5,0 m, 36 bairros têm pessoas afetadas, cerca de 127,6 mil pessoas, e a renda comprometida soma R$ 384,7 milhões por mês. Cinco bairros concentram 58% desse valor: Menino Deus (R$ 89,8 milhões), Sarandi (R$ 45,3 milhões), Humaitá (R$ 35,3 milhões), Floresta (R$ 27,2 milhões) e Cidade Baixa (R$ 26,0 milhões). Menino Deus lidera por combinar muitas pessoas afetadas com renda alta; Sarandi, por ter muitas pessoas afetadas, mesmo com renda menor. Essa relação entre exposição e renda é um ponto de partida para a Tela 2.